**Data Preprocessing**

Load in raw CSV files from MLS

In [28]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [29]:
folder = Path("raw")

csv_files = folder.glob("*.csv")

df = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

df.head()

/var/folders/mf/9m6y4nlx41z0kjyljz4sn3tc0000gn/T/ipykernel_82529/859079717.py:6: DtypeWarning: Columns (0: WaterfrontYN, 1: ElementarySchool, 2: BuilderName, 3: CoBuyerAgentFirstName, 4: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(file) for file in csv_files],
/var/folders/mf/9m6y4nlx41z0kjyljz4sn3tc0000gn/T/ipykernel_82529/859079717.py:6: DtypeWarning: Columns (0: latfilled, 1: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(file) for file in csv_files],
/var/folders/mf/9m6y4nlx41z0kjyljz4sn3tc0000gn/T/ipykernel_82529/859079717.py:6: DtypeWarning: Columns (0: WaterfrontYN) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(file) for file in csv_files],
/var/folders/mf/9m6y4nlx41z0kjyljz4sn3tc0000gn/T/ipykernel_82529/859079717.py:6: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
0,NaN,True,NaN,NaN,NaN,159000.0,555125771,deborah.potestio@c21selectgroup.com,2024-04-29,45000.0,...,NaN,NaN,95916,0.0,336283.2,NaN,False,False,NaN,NaN
1,NaN,True,NaN,NaN,NaN,144000.0,554271746,crchapman@sbcglobal.net,2024-04-26,78000.0,...,NaN,NaN,95966,0.0,101495.0,NaN,False,False,NaN,NaN
2,NaN,True,NaN,NaN,False,265000.0,543251400,michaelg@londonproperties.com,2024-04-02,250000.0,...,2.0,Fresno Unified,93726,0.0,8100.0,NaN,False,False,NaN,NaN
3,NaN,True,NaN,NaN,NaN,925000.0,539236677,chris.campbell@msn.com,2024-04-10,815000.0,...,NaN,NaN,92223,0.0,564988.0,NaN,False,False,NaN,NaN
4,NaN,True,NaN,NaN,NaN,25000.0,538449222,rrinder@sbcglobal.net,2024-04-06,15000.0,...,NaN,NaN,95966,0.0,6098.0,NaN,False,False,NaN,NaN


**Filter data**

In [30]:
# Filter data to only contain PropertyType="Residential" and PropertySubType="SingleFamilyResidence"
filtered_df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].reset_index(drop=True)

# Filter duplicate ListingKey
if "ListingKey" in filtered_df.columns:
    before = len(filtered_df)
    filtered_df = filtered_df.drop_duplicates(subset=["ListingKey"]).copy()
    print("Removed duplicate ListingKey rows:", before - len(filtered_df))

filtered_df[["PropertyType", "PropertySubType", "CloseDate", "ClosePrice"]].head()

Removed duplicate ListingKey rows: 276


,PropertyType,PropertySubType,CloseDate,ClosePrice
0,Residential,SingleFamilyResidence,2024-04-02,250000.0
1,Residential,SingleFamilyResidence,2024-04-30,413700.0
2,Residential,SingleFamilyResidence,2024-04-03,725000.0
3,Residential,SingleFamilyResidence,2024-04-12,600000.0
4,Residential,SingleFamilyResidence,2024-04-10,1810000.0


**Feature Set**

Numerical Features: LivingArea, BedroomsTotal, BathroomsTotalInteger, LotSizeSquareFeet, YearBuilt, GarageSpaces, ParkingTotal, Stories, Latitude, Longitude, AssociationFee

Categorical Features: City, PostalCode, CountyOrParish, MLSAreaMajor, HighSchoolDistrict, AssociationFeeFrequency

Boolean Features: ViewYN, PoolPrivateYN, NewConstructionYN, FireplaceYN, AttachedGarageYN

In [31]:
target_col = "ClosePrice"
date_col = "CloseDate"

# List of ID columns that actually exist within the DF
id_cols = [col for col in ["ListingKey", "ListingId", "source_file"] if col in filtered_df.columns]

numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "Latitude",
    "Longitude",
    "AssociationFee",
]


categorical_features = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "Levels",
    "HighSchoolDistrict",
    "AssociationFeeFrequency",
]

boolean_features = [
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN",
    "AttachedGarageYN",
]


# Safety step to make sure the notebook can run across different CSV's who may have missing features
numeric_features = [col for col in numeric_features if col in filtered_df.columns]
categorical_features = [col for col in categorical_features if col in filtered_df.columns]
boolean_features = [col for col in boolean_features if col in filtered_df.columns]

# Include PropertyType and PropertySubType for verification after cleaning to make sure it only contains the required property type.
model_input_cols = numeric_features + categorical_features + boolean_features + id_cols + [date_col, target_col, "PropertyType", "PropertySubType"]
model_df = filtered_df[model_input_cols]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Boolean/categorical features:", boolean_features)
print("Model input shape:", model_df.shape)


Numerical features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'ParkingTotal', 'Stories', 'Latitude', 'Longitude', 'AssociationFee']
Categorical features: ['City', 'PostalCode', 'CountyOrParish', 'MLSAreaMajor', 'Levels', 'HighSchoolDistrict', 'AssociationFeeFrequency']
Boolean/categorical features: ['ViewYN', 'PoolPrivateYN', 'NewConstructionYN', 'FireplaceYN', 'AttachedGarageYN']
Model input shape: (398881, 29)


**Date and Target Cleaning**

Missing CloseDate & missing/non-positive ClosePrice rows are dropped since they can not be used for supervised time-based modeling

In [32]:
model_df[date_col] = pd.to_datetime(model_df[date_col], errors="coerce")
model_df[target_col] = pd.to_numeric(model_df[target_col], errors="coerce")

before = len(model_df)
model_df = model_df[
    model_df[date_col].notna() &
    model_df[target_col].notna() &
    (model_df[target_col] > 0)
].copy()
print("Rows removed for invalid date/target:", before - len(model_df))

model_df["close_month"] = model_df[date_col].dt.to_period("M").dt.to_timestamp()

model_df.head()


Rows removed for invalid date/target: 2


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,ParkingTotal,Stories,Latitude,Longitude,...,NewConstructionYN,FireplaceYN,AttachedGarageYN,ListingKey,ListingId,CloseDate,ClosePrice,PropertyType,PropertySubType,close_month
0,1723.0,2.0,2.0,8100.0,1971.0,2.0,2.0,2.0,36.783028,-119.768450,...,False,True,True,543251400,FR21209594,2024-04-02,250000.0,Residential,SingleFamilyResidence,2024-04-01
1,2285.0,3.0,3.0,43560.0,1990.0,3.0,3.0,1.0,33.605045,-116.473489,...,False,True,True,519252621,SW21101127,2024-04-30,413700.0,Residential,SingleFamilyResidence,2024-04-01
2,1716.0,5.0,2.0,10557.0,1945.0,2.0,2.0,1.0,32.554950,-117.049710,...,NaN,False,True,492708527,PTP2001466,2024-04-03,725000.0,Residential,SingleFamilyResidence,2024-04-01
3,1600.0,3.0,3.0,8130.0,1958.0,2.0,4.0,1.0,32.742623,-116.988226,...,False,False,True,1073316150,240009799SD,2024-04-12,600000.0,Residential,SingleFamilyResidence,2024-04-01
4,2001.0,4.0,2.0,15600.0,1964.0,2.0,2.0,1.0,37.841644,-122.128606,...,False,True,True,1073301375,41058443,2024-04-10,1810000.0,Residential,SingleFamilyResidence,2024-04-01


**Numerical column cleaning**

Invalid numerical values are converted to missing, then a missing flag is added 

In [ ]:
for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

current_year = pd.Timestamp.today().year

# Impossible values (e.g., 0 Bathrooms) are treated as missing 
invalid_values = {}

if "LivingArea" in model_df.columns:
    invalid_values["LivingArea"] = model_df["LivingArea"] <= 0

if "BedroomsTotal" in model_df.columns:
    invalid_values["BedroomsTotal"] = model_df["BedroomsTotal"] <= 0

if "BathroomsTotalInteger" in model_df.columns:
    invalid_values["BathroomsTotalInteger"] = model_df["BathroomsTotalInteger"] <= 0

if "LotSizeSquareFeet" in model_df.columns:
    invalid_values["LotSizeSquareFeet"] = model_df["LotSizeSquareFeet"] <= 0

if "Stories" in model_df.columns:
    invalid_values["Stories"] = model_df["Stories"] <= 0

if "Latitude" in model_df.columns:
    invalid_values["Latitude"] = ~model_df["Latitude"].between(32, 42.5)

if "Longitude" in model_df.columns:
    invalid_values["Longitude"] = ~model_df["Longitude"].between(-125, -113)

invalid_summary = []
for col, mask in invalid_values.items():
    invalid_count = int(mask.fillna(False).sum())
    invalid_summary.append({"column": col, "invalid_values_set_to_missing": invalid_count})
    model_df.loc[mask.fillna(False), col] = np.nan

invalid_summary = pd.DataFrame(invalid_summary).sort_values("invalid_values_set_to_missing", ascending=False)
invalid_summary

,column,invalid_values_set_to_missing
6,Longitude,283
5,Latitude,253
1,BedroomsTotal,224
3,LotSizeSquareFeet,215
2,BathroomsTotalInteger,167
0,LivingArea,158
4,Stories,0


In [40]:
missing_flag_cols = []

for col in numeric_features:
    flag_col = f"{col}_missing_flag"
    model_df[flag_col] = model_df[col].isna().astype(int)
    missing_flag_cols.append(flag_col)

missing_report = pd.DataFrame({
    "missing_count": model_df[numeric_features].isna().sum(),
    "missing_rate": model_df[numeric_features].isna().mean()
}).sort_values("missing_rate", ascending=False)

missing_report


,missing_count,missing_rate
AssociationFee,122230,0.306434
Stories,54582,0.136838
GarageSpaces,14597,0.036595
ParkingTotal,7543,0.018910
LotSizeSquareFeet,7038,0.017644
LivingArea,368,0.000923
YearBuilt,308,0.000772
Longitude,283,0.000709
Latitude,253,0.000634
BathroomsTotalInteger,242,0.000607


**Categorical and Boolean Cleaning**

Categorical and boolean missing values are set to Unknown rather than being guessed

In [37]:
for col in categorical_features + boolean_features:
    model_df[col] = model_df[col].astype("object")
    model_df[col] = model_df[col].where(model_df[col].notna(), "Unknown")
    model_df[col] = model_df[col].astype(str).str.strip()
    model_df.loc[model_df[col].isin(["", "nan", "NaN", "None", "<NA>"]), col] = "Unknown"

model_df[categorical_features + boolean_features].head(5)

,City,PostalCode,CountyOrParish,MLSAreaMajor,Levels,HighSchoolDistrict,AssociationFeeFrequency,ViewYN,PoolPrivateYN,NewConstructionYN,FireplaceYN,AttachedGarageYN
0,Fresno,93726,Fresno,Unknown,Two,Fresno Unified,Unknown,True,False,False,True,True
1,Pinyon Pines,92561,Riverside,"326 - Pinyon Pines, Garner Valley",One,Palm Springs Unified,Unknown,True,False,False,True,True
2,San Ysidro,92173,San Diego,92173 - San Ysidro,One,Sweetwater Union,Unknown,True,False,Unknown,False,True
3,Spring Valley,91977,San Diego,91977 - Spring Valley,One,Unknown,Unknown,False,False,False,False,True
4,Moraga,94556,Contra Costa,Unknown,One,Unknown,Unknown,Unknown,False,False,True,True


**Save cleaned model-input data as a CSV**

Numerical imputation and categorical encoding are done after the export to avoid leaking test-set data to our training set

In [42]:
cleaned_csv_path = Path("outputs") / "cleaned_crmls_model_input.csv"
model_df.to_csv(cleaned_csv_path, index=False)

print("Saved cleaned CSV to:", cleaned_csv_path)
print("Cleaned shape:", model_df.shape)
model_df.head()

Saved cleaned CSV to: outputs/cleaned_crmls_model_input.csv
Cleaned shape: (398879, 41)


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,ParkingTotal,Stories,Latitude,Longitude,...,BedroomsTotal_missing_flag,BathroomsTotalInteger_missing_flag,LotSizeSquareFeet_missing_flag,YearBuilt_missing_flag,GarageSpaces_missing_flag,ParkingTotal_missing_flag,Stories_missing_flag,Latitude_missing_flag,Longitude_missing_flag,AssociationFee_missing_flag
0,1723.0,2.0,2.0,8100.0,1971.0,2.0,2.0,2.0,36.783028,-119.768450,...,0,0,0,0,0,0,0,0,0,0
1,2285.0,3.0,3.0,43560.0,1990.0,3.0,3.0,1.0,33.605045,-116.473489,...,0,0,0,0,0,0,0,0,0,0
2,1716.0,5.0,2.0,10557.0,1945.0,2.0,2.0,1.0,32.554950,-117.049710,...,0,0,0,0,0,0,0,0,0,0
3,1600.0,3.0,3.0,8130.0,1958.0,2.0,4.0,1.0,32.742623,-116.988226,...,0,0,0,0,0,0,0,0,0,1
4,2001.0,4.0,2.0,15600.0,1964.0,2.0,2.0,1.0,37.841644,-122.128606,...,0,0,0,0,0,0,0,0,0,1
